# Abertura P × Q pelo SAMP — Enel Distribuição São Paulo

Este notebook abre o faturamento por item tarifário e por família usando o **SAMP** como fonte
de quantidade e receita, em vez da extração do BW. É o par do `SP/BW/1-Entrega/Abertura_PxQ.ipynb`:
mesma tabela de tarifas, mesmas camadas, mesma lógica de blocos, mesma planilha de saída
(abas `Componentes` e `Familias`).

| Bloco | O que é |
|---|---|
| **1 — Calculado** | quantidade do SAMP × tarifa homologada, aberto nos componentes da tarifa. |
| **2 — Ajustado** | os mesmos itens ajustados para fechar na receita declarada no SAMP. É **a abertura**. |

**Por que o SAMP.** O SAMP (Sistema de Acompanhamento de Informações de Mercado para Regulação
Econômica, REN ANEEL 1.003/2022) é o mercado faturado que a distribuidora declara à ANEEL. Ele traz
o que o BW não traz:

- o **detalhe do consumidor livre** (APE, fonte incentivada, ERC), o que permite aplicar a tarifa
  APE, sem encargos, a quem é APE;
- as **faixas dos benefícios sociais** declaradas (Tarifa Social e Desconto Social, faixas 01 e 02),
  sem precisar estimá-las pela receita;
- o **percentual de desconto** declarado de cada linha de fonte incentivada e de geradora;
- a **energia TUSD e a energia TE separadas**, o que dispensa as regras do BW sobre qual lado do
  lançamento carrega a quantidade.

O que o SAMP não traz: receita separada TUSD × TE nem receita por componente. Por isso a abertura
continua sendo quantidade × tarifa, ajustada para fechar na receita.

Contexto e resultados: `SAMP/projeto_samp.md`. Regras do motor: seção 4.1 de lá.

---
## Passo 0 — Parâmetros

Tudo que muda de um mês para o outro está aqui. O resto do notebook não precisa ser editado.

In [ ]:
import os
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------- pastas
ROOT = Path(os.environ.get("PXQ_ROOT", r"C:\python_projects\claude\p_x_q"))

ARQ_SAMP        = ROOT / "SAMP" / "data" / "samp-2026.parquet"    # SAMP do ano, todas as distribuidoras
ARQ_TARIFA_LONG = ROOT / "SP" / "BW" / "tbl" / "tarifas" / "tarifa_long.csv"  # mesma tabela do PxQ BW

# ---------------------------------------------------------------- periodo
REFERENCIA = "2026-03"

# ---------------------------------------------------------------- distribuidora
CNPJ_DISTRIBUIDORA = "61695227000193"      # Enel SP (ELETROPAULO), 14 digitos, como texto

SAIDA = ROOT / "SAMP" / "1-Entrega" / "pxq" / f"pxq_samp_{REFERENCIA}.xlsx"

print("SAMP     :", ARQ_SAMP.name)
print("referencia:", REFERENCIA)
print("saida    :", SAIDA.name)

---
## Passo 1 — Ler o SAMP

O arquivo é **longo**: uma linha por combinação de atributos × métrica (`DscDetalheMercado`,
24 métricas) com o valor em `VlrMercado`. Traz todas as distribuidoras do ano.

Dois cuidados na leitura:

- **CNPJ como texto de 14 dígitos.** `NumCNPJAgenteDistribuidora` vem como inteiro e perde o zero à
  esquerda. Comparar direto com a string não acha nada (foi o que deixou o `samp-SP-2026.parquet`
  vazio). O filtro completa com zeros antes de comparar.
- **Competência como data do mês.** Uso `DatCompetencia`, a mesma convenção das demais bases (o BW
  não separa refaturamento).

Os nove atributos que definem o fornecimento são renomeados para nomes curtos; vazio vira
`Não se aplica`, como na tabela de tarifas.

**ERC fora do cálculo (decisão de 29/09/2026).** As linhas com detalhe `ERC` (Encargo de Energia de
Reserva — no SAMP de SP, só demanda do livre A2 Azul) são retiradas aqui. A regra de cobrança não é
conhecida: a partir de mai/26 a mesma demanda passa a render 78% menos. O valor retirado é informado
na saída desta célula. O PxQ pelo BW também não tem ERC (ele não passa pelo BW).

In [ ]:
NSA = "Não se aplica"

# SAMP attribute -> short name used in this notebook
ATRIBUTOS_SAMP = {
    "NomTipoMercado": "TIPO_MERCADO", "DscOpcaoEnergia": "OPCAO",
    "DscSubGrupoTarifario": "SUBGRUPO", "DscModalidadeTarifaria": "MODALIDADE",
    "DscClasseConsumoMercado": "CLASSE_SAMP", "DscSubClasseConsumidor": "SUBCLASSE_SAMP",
    "DscDetalheConsumidor": "DETALHE_SAMP", "DscPostoTarifario": "POSTO",
    "NomAgenteAcessante": "AGENTE",
}
GRUPO_SAMP = list(ATRIBUTOS_SAMP.values())

samp = pd.read_parquet(ARQ_SAMP)
samp = samp[samp["NumCNPJAgenteDistribuidora"].astype(str).str.zfill(14) == CNPJ_DISTRIBUIDORA]
competencias = sorted(pd.to_datetime(samp["DatCompetencia"]).dt.strftime("%Y-%m").unique())
samp = samp[pd.to_datetime(samp["DatCompetencia"]).dt.strftime("%Y-%m") == REFERENCIA].copy()
assert len(samp), f"SAMP sem linhas para {REFERENCIA} (competencias disponiveis: {competencias})"

samp = samp.rename(columns=ATRIBUTOS_SAMP)
for c in GRUPO_SAMP:
    samp[c] = samp[c].fillna(NSA).astype(str).str.strip()

# ERC (reserve energy charge) is left out until its billing rule is known
erc = samp["DETALHE_SAMP"] == "ERC"
ERC_FORA = samp.loc[erc].pivot_table(index="DscDetalheMercado", values="VlrMercado", aggfunc="sum")["VlrMercado"]
samp = samp.loc[~erc].copy()

print(f"competencias no arquivo: {', '.join(competencias)}")
print(f"{len(samp)} linhas do SAMP em {REFERENCIA} | {samp['DscDetalheMercado'].nunique()} metricas")
if len(ERC_FORA):
    print("ERC retirado: " + " | ".join(f"{k} {v:,.2f}" for k, v in ERC_FORA.items()
                                       if k in ("Demanda Faturada (kW)", "Receita Demanda (R$)", "Receita Energia (R$)")))

---
## Passo 2 — Uma linha por fornecimento

O formato longo vira uma linha por **grupo** (os nove atributos) com as métricas em colunas.
Das 24 métricas, entram estas:

| Métrica do SAMP | Coluna | Tratamento |
|---|---|---|
| Demanda Faturada (kW) | `DEMANDA_KW` | soma |
| Energia TUSD (kWh) / Energia TE (kWh) | `E_TUSD_MWH` / `E_TE_MWH` | soma, em MWh (a tarifa de energia é R$/MWh) |
| Receita Energia (R$) + Receita Demanda (R$) | `RECEITA_BRL` | soma — é o alvo do bloco 2 |
| Receita energia compensada (R$) / Energia compensada (kWh) | — | ver abaixo |
| Desconto Demanda % / Desconto Energia TUSD % | `DESC_DEM_PCT` / `DESC_E_PCT` | média — é uma taxa, não se soma |

**Energia compensada.** No GD II, o Mercado Receita cria uma linha própria, *Sistema de
Compensação GD II - SCEE*, com a energia compensada e a receita dela. Faço o mesmo aqui, com detalhe
`SCEE` (que tem tarifa própria no PCAT). Nos demais tipos de mercado, a receita compensada é somada à
receita da linha.

Ficam de fora, por não terem tarifa de componente: ultrapassagem de demanda, bandeiras, tributos e
unidades consumidoras. São memos do SAMP.

In [ ]:
# SAMP metric -> column; rates (_PCT) are averaged, everything else is summed
METRICA = {
    "Demanda Faturada (kW)": "DEMANDA_KW", "Energia TUSD (kWh)": "E_TUSD_MWH",
    "Energia TE (kWh)": "E_TE_MWH", "Receita Energia (R$)": "RECEITA_BRL",
    "Receita Demanda (R$)": "RECEITA_BRL",
    "Receita energia compensada (R$)": "RECEITA_COMP_BRL",
    "Energia compensada (kWh)": "E_COMP_MWH",
    "Desconto Demanda %": "DESC_DEM_PCT", "Desconto Energia TUSD %": "DESC_E_PCT",
}
ESCALA = {"E_TUSD_MWH": 1e-3, "E_TE_MWH": 1e-3, "E_COMP_MWH": 1e-3}     # kWh -> MWh

samp["MEDIDA"] = samp["DscDetalheMercado"].map(METRICA)
x = samp[samp["MEDIDA"].notna()].copy()
x["VALOR"] = x["VlrMercado"] * x["MEDIDA"].map(ESCALA).fillna(1.0)
aditivas = x[~x["MEDIDA"].str.endswith("_PCT")].pivot_table(
    index=GRUPO_SAMP, columns="MEDIDA", values="VALOR", aggfunc="sum", fill_value=0.0)
taxas = x[x["MEDIDA"].str.endswith("_PCT")].pivot_table(
    index=GRUPO_SAMP, columns="MEDIDA", values="VALOR", aggfunc="mean")
base = aditivas.join(taxas, how="left").reset_index()
for c in ["DEMANDA_KW", "E_TUSD_MWH", "E_TE_MWH", "RECEITA_BRL", "RECEITA_COMP_BRL", "E_COMP_MWH"]:
    if c not in base.columns:
        base[c] = 0.0
for c in ["DESC_DEM_PCT", "DESC_E_PCT"]:
    if c not in base.columns:
        base[c] = np.nan

# GD II compensated energy becomes its own row (the MR "GD II - SCEE" line)
gd2 = base["TIPO_MERCADO"] == "Sistema de Compensação GD II"
scee = base[gd2 & ((base["E_COMP_MWH"] != 0) | (base["RECEITA_COMP_BRL"] != 0))].copy()
scee["TIPO_MERCADO"] = "Sistema de Compensação GD II - SCEE"
scee["DETALHE_SAMP"] = "SCEE"
scee["E_TUSD_MWH"] = scee["E_COMP_MWH"]
scee["E_TE_MWH"] = scee["E_COMP_MWH"]
scee["DEMANDA_KW"] = 0.0
scee["RECEITA_BRL"] = scee["RECEITA_COMP_BRL"]
scee[["DESC_DEM_PCT", "DESC_E_PCT"]] = np.nan
base["RECEITA_BRL"] = base["RECEITA_BRL"] + np.where(gd2, 0.0, base["RECEITA_COMP_BRL"])
base = pd.concat([base, scee], ignore_index=True).drop(columns=["RECEITA_COMP_BRL", "E_COMP_MWH"])

print(f"{len(base)} grupos ({len(scee)} linhas SCEE criadas)")
print(f"receita SAMP R$ {base['RECEITA_BRL'].sum():,.2f} | energia TUSD {base['E_TUSD_MWH'].sum():,.3f} MWh"
      f" | energia TE {base['E_TE_MWH'].sum():,.3f} MWh | demanda {base['DEMANDA_KW'].sum():,.0f} kW")

### Passo 2b — Conferência de qualidade: receita repetida do mês anterior

Em jun/26 o SAMP repete exatamente a receita de mai/26 no B1 Residencial Regular (só a energia
muda) — é dado copiado, não faturado. A conferência abaixo compara, grupo a grupo, a receita do mês
com a do mês anterior e avisa quando um grupo relevante tem receita **idêntica** com energia
diferente. Não interrompe o cálculo: o aviso é para não usar o mês sem confirmar com a área.

In [ ]:
anterior = (pd.Period(REFERENCIA, freq="M") - 1).strftime("%Y-%m")
ant = pd.read_parquet(ARQ_SAMP)
ant = ant[(ant["NumCNPJAgenteDistribuidora"].astype(str).str.zfill(14) == CNPJ_DISTRIBUIDORA)
          & (pd.to_datetime(ant["DatCompetencia"]).dt.strftime("%Y-%m") == anterior)]
RECEITA_REPETIDA = pd.DataFrame()
if len(ant):
    ant = ant.rename(columns=ATRIBUTOS_SAMP)
    for c in GRUPO_SAMP:
        ant[c] = ant[c].fillna(NSA).astype(str).str.strip()
    def _pivota(df):
        m = df[df["DscDetalheMercado"].isin(["Receita Energia (R$)", "Energia TUSD (kWh)"])]
        return m.pivot_table(index=GRUPO_SAMP, columns="DscDetalheMercado", values="VlrMercado",
                             aggfunc="sum")
    a, b = _pivota(samp), _pivota(ant)
    j = a.join(b, how="inner", rsuffix=" ant").dropna()
    rep = j[(j["Receita Energia (R$)"] == j["Receita Energia (R$) ant"])
            & (j["Receita Energia (R$)"].abs() > 1e6)
            & (j["Energia TUSD (kWh)"] != j["Energia TUSD (kWh) ant"])]
    RECEITA_REPETIDA = rep.reset_index()
    if len(rep):
        print(f"ATENCAO: {len(rep)} grupo(s) com receita identica a {anterior} e energia diferente "
              f"(R$ {rep['Receita Energia (R$)'].sum():,.2f}) - confirmar o SAMP antes de usar o mes:")
        for _, r in RECEITA_REPETIDA.iterrows():
            print(f"   {r['TIPO_MERCADO']} | {r['SUBGRUPO']} {r['MODALIDADE']} | {r['SUBCLASSE_SAMP']} | "
                  f"R$ {r['Receita Energia (R$)']:,.2f}")
    else:
        print(f"receita sem repeticao em relacao a {anterior}")
else:
    print(f"sem SAMP de {anterior} para comparar")

---
## Passo 3 — As tarifas homologadas

A mesma tabela central do PxQ pelo BW, `SP/BW/tbl/tarifas/tarifa_long.csv` (gerada por
`SP/BW/3-Py/1-Entrega/extrai_pcat.py`): uma linha por vigência × camada × chave, com os componentes em
colunas e `VIGENCIA_INICIO`/`VIGENCIA_FIM`.

- **Camadas:** TA (tarifa de aplicação) = BE (base econômica) + CVA + FIN (financeiros).
- **Vigência ponderada por dias:** o peso de cada vigência é a sobreposição, em dias, com o mês de
  referência. Uma chave ausente em uma das vigências tem o peso redistribuído entre as presentes.
- **Chave com separador.** Aqui a chave é `SUBGRUPO|MODALIDADE|CLASSE|SUBCLASSE|DETALHE|ACESSANTE|POSTO|UNIDADE`,
  com o posto por extenso, como no PCAT. Não é preciso reproduzir o código abreviado do SAP, porque o
  SAMP já escreve o posto por extenso.
- **Família de cada componente:** o mesmo mapa do notebook do BW. Componente novo na tabela sem
  família faz o notebook parar.

In [ ]:
CAMADAS = ["TA", "BE", "CVA", "FIN"]

# component -> family (same map as the BW notebook)
FAMILIA = {
    "FIO A": ["TUSD_RB", "TUSD_FR", "TUSD_CCT", "TUSD_CCD", "TUSD_CUSD", "TUSDG_T",
              "TE_TRANSPORTE_ITAIPU", "TE_TUST_ITAIPU", "TE_TUST_CI"],
    "FIO B": ["TUSD_FioB"],
    "ENCARGOS": ["TUSD_CDE_COVID", "TUSD_TFSEE", "TUSD_PeD", "TUSD_ONS", "TUSD_CCC", "TUSD_CDE",
                 "TUSD_PROINFA", "Liminar1", "TUSDG_ONS", "TE_CDE_COVID", "TE_CDE_ELET",
                 "TE_CDE_GD", "TE_PeD", "TE_ESSERR", "TE_CFURH"],
    "PERDAS": ["TUSD_PT", "TUSD_Per_RB_D", "TUSD_PNT", "TE_Per_RB"],
    "ENERGIA": ["TE_ENERGIA", "TE_ANGRA"],
    "RI": ["TUSD_RI"],
    "OUTROS": ["TUSD Subsidio", "TUSD BENEFICIO_L14299", "TUSD Outros",
               "TE Subsidio", "TE BENEFICIO_L14299"],
}
FAMILIAS = list(FAMILIA)

t = pd.read_csv(ARQ_TARIFA_LONG, sep=";", decimal=",", encoding="utf-8-sig")
t["VIGENCIA_INICIO"] = pd.to_datetime(t["VIGENCIA_INICIO"]).dt.date
t["VIGENCIA_FIM"] = pd.to_datetime(t["VIGENCIA_FIM"]).dt.date
ATRIB_CHAVE = ["SUBGRUPO", "MODALIDADE", "CLASSE", "SUBCLASSE", "DETALHE", "ACESSANTE", "POSTO", "UNIDADE"]
nao_comp = set(ATRIB_CHAVE) | {"ANO_VIGENCIA", "VIGENCIA_INICIO", "VIGENCIA_FIM", "CAMADA", "ORIGEM",
                               "Total TUSD", "Total TE"}
COMPONENTES = [c for c in t.columns if c not in nao_comp]
faltam = [c for c in COMPONENTES if c not in {x for l in FAMILIA.values() for x in l}]
assert not faltam, f"componentes sem familia: {faltam}"
COMP_TE = [c for c in COMPONENTES if c.startswith("TE")]
COMP_TUSD = [c for c in COMPONENTES if c not in COMP_TE]
t["CHAVE"] = t[ATRIB_CHAVE].astype(str).agg("|".join, axis=1)

# weight of each vintage = calendar overlap with the reference month
ini = dt.date(int(REFERENCIA[:4]), int(REFERENCIA[5:7]), 1)
fim = (pd.Timestamp(ini) + pd.offsets.MonthEnd(0)).date()
dias_mes = (fim - ini).days + 1
VIGENCIAS_NO_MES = []
for _, v in t[["ANO_VIGENCIA", "VIGENCIA_INICIO", "VIGENCIA_FIM"]].drop_duplicates().iterrows():
    d = (min(v["VIGENCIA_FIM"], fim) - max(v["VIGENCIA_INICIO"], ini)).days + 1
    if d > 0:
        VIGENCIAS_NO_MES.append((int(v["ANO_VIGENCIA"]), d / dias_mes))
assert VIGENCIAS_NO_MES, f"nenhuma vigencia cobre {REFERENCIA}"

TARIFA = {}
for cam in CAMADAS:
    tc = t[t["CAMADA"] == cam].set_index(["ANO_VIGENCIA", "CHAVE"])[COMPONENTES]
    soma = pd.concat([tc.xs(a, level=0) * p for a, p in VIGENCIAS_NO_MES]).groupby(level=0).sum()
    # keys missing in one vintage: weight redistributed over the ones present
    presente = pd.concat([pd.Series(p, index=tc.xs(a, level=0).index) for a, p in VIGENCIAS_NO_MES]
                         ).groupby(level=0).sum()
    TARIFA[cam] = soma.div(presente, axis=0)

print(f"{len(COMPONENTES)} componentes | {len(TARIFA['TA'])} chaves | vigencias em {REFERENCIA}: "
      + ", ".join(f"{a} ({100 * p:.1f}%)" for a, p in VIGENCIAS_NO_MES))

---
## Passo 4 — Montar a chave tarifária

Os atributos do SAMP não têm o mesmo vocabulário do PCAT em quatro posições. A tradução:

| Atributo da chave | Regra |
|---|---|
| **CLASSE** | derivada do subgrupo: B1 = Residencial, B2 = Rural, B4 = Iluminação pública, demais = Não se aplica |
| **SUBCLASSE** | só no B1 e no B4. B1: Tarifa Social e baixa renda legado = `Baixa Renda`; Desconto Social = a própria faixa; demais = `Residencial`. B4 = a do SAMP. Demais subgrupos = Não se aplica |
| **DETALHE** | só `APE` e `SCEE` têm tarifa própria. Fonte incentivada, ERC, <500, IRRIG. = Não se aplica |
| **ACESSANTE** | só geração, distribuição e suprimento, pelo de-para abaixo; demais = Não se aplica |

**APE com tarifa própria.** É aqui que o SAMP corrige o PxQ do BW: o autoprodutor por equiparação
recebe a tarifa APE do PCAT, cuja TUSD de energia não tem encargos. No BW não há como identificar o
APE (`SP/BW/README.md` 4.10).

**Chaves de reserva**, registradas nas colunas `REGRA_E` / `REGRA_D` para rastreio:

1. posto `Não se aplica` → `Fora ponta` (energia Azul/APE lançada sem posto; a TUSD de energia é igual
   nos postos);
2. posto informado → `Não se aplica` (energia da Ceris);
3. acessante nomeado → genérico (Tecipar usa a chave A4 Geração).

Grupo sem nenhuma chave fica **sem tarifa** e sua receita vai para o resíduo não ajustável.

In [ ]:
# SAMP accessing agent -> PCAT ACESSANTE (Tecipar has no row of its own: generic key)
ACESSANTE_PCAT = {
    "ELEKTRO REDES S.A.": "Neoenergia Elektro",
    "COMPANHIA PIRATININGA DE FORCA E LUZ": "CPFL Piratininga",
    "COOPERATIVA DE ELETRIFICAÇÃO DA REGIÃO DE ITAPECERICA DA SERRA": "Ceris",
    "Rasgão": "PCH RASGAO",
    "Nova Piratininga (Antiga Fernando Gasparian)": "UTE FERNANDO GASPARIAN (N.PIRAT)",
    "São João Biogás": "UTE SAO JOAO BIOGAS",
    "Pirapora": "PCH PIRAPORA",
}


def classe_pcat(subgrupo):
    return {"B1": "Residencial", "B2": "Rural", "B4": "Iluminação pública"}.get(subgrupo, NSA)


def subclasse_pcat(r):
    s, sg = r["SUBCLASSE_SAMP"], r["SUBGRUPO"]
    if sg == "B4":
        return s
    if sg != "B1":
        return NSA
    if "Tarifa Social" in s or "baixa renda" in s:
        return "Baixa Renda"
    if "Desconto Social" in s:
        return s
    return "Residencial"


base["CLASSE"] = base["SUBGRUPO"].map(classe_pcat)
base["SUBCLASSE"] = base.apply(subclasse_pcat, axis=1)
base["DETALHE"] = base["DETALHE_SAMP"].where(base["DETALHE_SAMP"].isin(["APE", "SCEE"]), NSA)
base["ACESSANTE"] = np.where(base["OPCAO"].isin(["GERAÇÃO", "DISTRIBUIÇÃO", "SUPRIMENTO"]),
                             base["AGENTE"].map(ACESSANTE_PCAT).fillna(NSA), NSA)


def acha_chave(r, unidade):
    '''Exact key first, then the fallbacks. Returns (key, rule).'''
    tentativas = [(r["POSTO"], r["ACESSANTE"], "exata")]
    if r["POSTO"] == NSA:
        tentativas.append(("Fora ponta", r["ACESSANTE"], "posto NSA -> Fora ponta"))
    else:
        tentativas.append((NSA, r["ACESSANTE"], "posto -> NSA"))
    if r["ACESSANTE"] != NSA:
        tentativas.append((r["POSTO"], NSA, "acessante generico"))
    for posto, acessante, regra in tentativas:
        k = "|".join([r["SUBGRUPO"], r["MODALIDADE"], r["CLASSE"], r["SUBCLASSE"],
                      r["DETALHE"], acessante, posto, unidade])
        if k in TARIFA["TA"].index:
            return k, regra
    return None, "sem tarifa"


tem_energia = (base["E_TUSD_MWH"] != 0) | (base["E_TE_MWH"] != 0)
base[["CHAVE_E", "REGRA_E"]] = base.apply(
    lambda r: pd.Series(acha_chave(r, "MWh") if tem_energia[r.name] else (None, "")), axis=1)
base[["CHAVE_D", "REGRA_D"]] = base.apply(
    lambda r: pd.Series(acha_chave(r, "kW") if r["DEMANDA_KW"] != 0 else (None, "")), axis=1)

regras = pd.concat([base["REGRA_E"], base["REGRA_D"]])
print("chaves:", regras[regras != ""].value_counts().to_dict())
sem = base[(tem_energia & base["CHAVE_E"].isna()) | ((base["DEMANDA_KW"] != 0) & base["CHAVE_D"].isna())]
if len(sem):
    print(f"ATENCAO: {len(sem)} grupo(s) sem tarifa - receita R$ {sem['RECEITA_BRL'].sum():,.2f}")
    print(sem[["OPCAO", "SUBGRUPO", "MODALIDADE", "SUBCLASSE_SAMP", "DETALHE_SAMP", "POSTO",
               "AGENTE", "RECEITA_BRL"]].to_string(index=False))

---
## Passo 5 — Multiplicar: o bloco 1

Para cada grupo, componente e camada:

```
componente TUSD = tarifa da chave de energia × energia TUSD  +  tarifa da chave de demanda × demanda
componente TE   = tarifa da chave de energia × energia TE
```

Como o SAMP separa energia TUSD e energia TE, a regra é a mesma para todos: o cativo tem as duas
iguais, o livre tem TE = 0 e o suprimento tem a TUSD com PROINFA. Não é preciso a tabela de
"quais parcelas incidem" do notebook do BW.

**Benefícios sociais pela faixa declarada** (sem estimativa pela receita, ao contrário do BW):

- **Tarifa Social faixa 01** (até 80 kWh): redução de 100% → valor zero. A receita do SAMP também é
  zero. O Mercado Receita mantém a tarifa cheia e lança o benefício como subsídio, por isso o
  bloco 1 fica abaixo do `Total com Tar de aplicação` do MR — é esperado.
- **Tarifa Social faixa 02 e baixa renda legado**: chave `Baixa Renda` com a isenção de encargos
  de `SP/BW/README.md` 4.5 (TUSD_CDE, TUSD_PROINFA, TUSD_CDE_COVID, TE_CDE_GD zerados).
- **Desconto Social faixa 01 e 02**: chaves próprias.

In [ ]:
# charge exemption of the social benefits (SP/BW/README.md 4.5)
ISENCAO_BAIXA_RENDA = ["TUSD_CDE", "TUSD_PROINFA", "TUSD_CDE_COVID", "TE_CDE_GD"]
base["ISENCAO_BR"] = base["SUBCLASSE"] == "Baixa Renda"
base["TARIFA_SOCIAL_F01"] = base["SUBCLASSE_SAMP"].str.contains("Tarifa Social – faixa 01")


def valoriza(camada):
    '''Quantity x tariff for one layer, one column per component.'''
    te = TARIFA[camada].reindex(base["CHAVE_E"]).fillna(0.0)
    te.index = base.index
    td = TARIFA[camada].reindex(base["CHAVE_D"]).fillna(0.0)
    td.index = base.index
    v = pd.DataFrame(0.0, index=base.index, columns=COMPONENTES)
    v[COMP_TUSD] = (te[COMP_TUSD].mul(base["E_TUSD_MWH"], axis=0)
                    + td[COMP_TUSD].mul(base["DEMANDA_KW"], axis=0))
    v[COMP_TE] = te[COMP_TE].mul(base["E_TE_MWH"], axis=0)
    v.loc[base["ISENCAO_BR"], ISENCAO_BAIXA_RENDA] = 0.0
    v.loc[base["TARIFA_SOCIAL_F01"], :] = 0.0
    return v


BLOCO1 = {c: valoriza(c) for c in CAMADAS}
base["BLOCO1_TA"] = BLOCO1["TA"].sum(axis=1)
base["DIFERENCA_BRL"] = base["RECEITA_BRL"] - base["BLOCO1_TA"]
base["PCT_AJUSTE"] = np.where(base["BLOCO1_TA"] != 0,
                              base["DIFERENCA_BRL"] / base["BLOCO1_TA"].replace(0, np.nan), np.nan)

print(f"bloco 1 (TA) R$ {base['BLOCO1_TA'].sum():,.2f} | receita SAMP R$ {base['RECEITA_BRL'].sum():,.2f}"
      f" | diferenca R$ {base['DIFERENCA_BRL'].sum():,.2f}")

---
## Passo 6 — O ajuste: o bloco 2

Mesma lógica do notebook do BW, em duas passadas:

1. **1ª passada — desconto de fonte incentivada sobre Fio A + Fio B.** Só nos grupos do grupo A que
   têm desconto declarado no SAMP (`Desconto Demanda %` ou `Desconto Energia TUSD %`): regime
   `fio`. O percentual é o que leva o bloco 1 à receita descontando apenas o fio:

   ```
   % desconto no fio = (bloco 1 − receita) / (Fio A + Fio B do bloco 1), limitado a [0, 1]
   ```

   Encargos, perdas e energia ficam intactos, porque o desconto legal não incide sobre eles.
2. **2ª passada — fechamento uniforme.** Todos os grupos (regime `uniforme` e o que sobrou no regime
   `fio`) são escalados por igual para fechar na receita do SAMP.

Grupo com receita e sem nada para escalar (bloco 1 zero) não fecha: a receita fica como **resíduo
não ajustável**. Hoje é o caso da TE de suprimento da Ceris, que não tem tarifa na tabela.

In [ ]:
COLS_FIO = FAMILIA["FIO A"] + FAMILIA["FIO B"]

tem_desconto = base["DESC_DEM_PCT"].notna() | base["DESC_E_PCT"].notna()
base["REGIME"] = np.where(tem_desconto & base["SUBGRUPO"].str.startswith("A"), "fio", "uniforme")
em_fio = base["REGIME"] == "fio"

# 1st pass: discount only on Fio A + Fio B, rows with a declared discount
fio_ta = BLOCO1["TA"][COLS_FIO].sum(axis=1)
falta = base["BLOCO1_TA"] - base["RECEITA_BRL"]
desc = np.where(em_fio & (fio_ta != 0), falta / fio_ta.replace(0, np.nan), 0.0)
base["DESC_FIO"] = np.clip(np.nan_to_num(desc), 0.0, 1.0)
BLOCO2 = {}
for cam in CAMADAS:
    m = BLOCO1[cam].copy()
    m[COLS_FIO] = m[COLS_FIO].mul(1.0 - base["DESC_FIO"], axis=0)
    BLOCO2[cam] = m
parcial = BLOCO2["TA"].sum(axis=1)
base["AJUSTE_1A_BRL"] = parcial - base["BLOCO1_TA"]

# 2nd pass: uniform scaling to close on the SAMP revenue
fator2 = np.nan_to_num(np.where(parcial != 0, base["RECEITA_BRL"] / parcial.replace(0, np.nan), 0.0))
BLOCO2 = {c: BLOCO2[c].mul(fator2, axis=0) for c in CAMADAS}
base["BLOCO2_TA"] = BLOCO2["TA"].sum(axis=1)
base["AJUSTE_2A_BRL"] = base["BLOCO2_TA"] - parcial
base["RESIDUO_NAO_AJUSTAVEL"] = base["RECEITA_BRL"] - base["BLOCO2_TA"]

print(f"grupos em regime fio: {int(em_fio.sum())} | uniforme: {int((~em_fio).sum())}")
print(f"1a passada (desconto sobre o fio) R$ {base['AJUSTE_1A_BRL'].sum():,.2f}")
print(f"2a passada (fechamento)           R$ {base['AJUSTE_2A_BRL'].sum():,.2f}")
print(f"bloco 2 (TA)                      R$ {base['BLOCO2_TA'].sum():,.2f}")

---
## Passo 7 — Conferências

Todas têm de passar antes de a planilha ser usada.

1. Bloco 2 = receita do SAMP em todos os grupos ajustáveis.
2. TA = BE + CVA + FIN nos dois blocos.
3. Soma das famílias = soma dos componentes nos dois blocos.
4. Receita do arquivo = receita da base (nada se perdeu no agrupamento).

O **resíduo não ajustável** é informado à parte, em reais, junto com os grupos que o formam.

In [ ]:
ok = True
ajustavel = base["RESIDUO_NAO_AJUSTAVEL"].abs() < 0.01
d1 = (base.loc[ajustavel, "BLOCO2_TA"] - base.loc[ajustavel, "RECEITA_BRL"]).abs().max()
print(f"1. bloco 2 x receita, grupos ajustaveis ..... diferenca maxima {d1:.6f}")
ok &= d1 < 0.01

d2 = max((b["TA"] - b["BE"] - b["CVA"] - b["FIN"]).abs().max().max() for b in (BLOCO1, BLOCO2))
print(f"2. TA = BE + CVA + FIN, blocos 1 e 2 ....... diferenca maxima {d2:.10f}")
ok &= d2 < 1e-5

d3 = max(abs(sum(b["TA"][l].sum().sum() for l in FAMILIA.values()) - b["TA"].sum().sum())
         for b in (BLOCO1, BLOCO2))
print(f"3. familias x componentes .................. diferenca maxima {d3:.6f}")
ok &= d3 < 0.01

receita_arquivo = samp.loc[samp["DscDetalheMercado"].isin(
    ["Receita Energia (R$)", "Receita Demanda (R$)", "Receita energia compensada (R$)"]), "VlrMercado"].sum()
d4 = abs(receita_arquivo - base["RECEITA_BRL"].sum())
print(f"4. receita do SAMP x receita da base ........ diferenca {d4:.6f}")
ok &= d4 < 0.01

residuo = base.loc[~ajustavel]
print(f"\nresiduo nao ajustavel ({len(residuo)} grupos) R$ {residuo['RESIDUO_NAO_AJUSTAVEL'].sum():,.2f}")
for _, r in residuo.iterrows():
    print(f"   {r['OPCAO']} | {r['SUBGRUPO']} {r['MODALIDADE']} | {r['AGENTE']} | "
          f"R$ {r['RESIDUO_NAO_AJUSTAVEL']:,.2f}")

print("\nTODAS AS CONFERENCIAS PASSARAM" if ok else "\nHA CONFERENCIA COM FALHA - NAO USAR A PLANILHA")
assert ok

---
## Passo 8 — Gerar a planilha

Duas abas, com a mesma identificação nas duas, no mesmo desenho da planilha do BW:

| Aba | Bloco 1 | Bloco 2 |
|---|---|---|
| **Componentes** | os componentes da tarifa de aplicação | os mesmos, ajustados, mais os totais ajustados de TA, BE, CVA e FIN |
| **Familias** | Fio A, Fio B, Encargos, Perdas, Energia, RI e Outros, nas 4 camadas | as mesmas, ajustadas |

Entre os dois blocos ficam o regime de ajuste, o % de desconto no fio, os ajustes da 1ª e da 2ª
passada e o resíduo não ajustável.

A identificação traz os atributos do SAMP (como vieram) e os da chave tarifária (como foram
traduzidos), mais a regra de chave usada. Grupo sem tarifa aparece com `Tarifável?` = `SEM TARIFA`.

Essa planilha é a entrada dos checks em `SAMP/1-Entrega/checks/py/`.

In [ ]:
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

base["SOURCE"] = "SAMP"
base["TARIFAVEL_TXT"] = np.where(
    (tem_energia & base["CHAVE_E"].isna()) | ((base["DEMANDA_KW"] != 0) & base["CHAVE_D"].isna()),
    "SEM TARIFA", "SIM")

IDENTIFICACAO = ["SOURCE", "TIPO_MERCADO", "OPCAO", "SUBGRUPO", "MODALIDADE", "CLASSE_SAMP",
                 "SUBCLASSE_SAMP", "DETALHE_SAMP", "POSTO", "AGENTE",
                 "CLASSE", "SUBCLASSE", "DETALHE", "ACESSANTE",
                 "CHAVE_E", "REGRA_E", "CHAVE_D", "REGRA_D",
                 "E_TUSD_MWH", "E_TE_MWH", "DEMANDA_KW", "DESC_DEM_PCT", "DESC_E_PCT",
                 "RECEITA_BRL", "BLOCO1_TA", "DIFERENCA_BRL", "PCT_AJUSTE", "TARIFAVEL_TXT"]
ENTRE_BLOCOS = ["REGIME", "DESC_FIO", "AJUSTE_1A_BRL", "AJUSTE_2A_BRL", "RESIDUO_NAO_AJUSTAVEL"]
ROTULO = {"SOURCE": "Source", "TIPO_MERCADO": "Tipo de mercado", "OPCAO": "Opção",
          "SUBGRUPO": "Subgrupo", "MODALIDADE": "Modalidade", "CLASSE_SAMP": "Classe (SAMP)",
          "SUBCLASSE_SAMP": "Subclasse (SAMP)", "DETALHE_SAMP": "Detalhe (SAMP)",
          "POSTO": "Posto", "AGENTE": "Agente acessante (SAMP)",
          "CLASSE": "Classe (tarifa)", "SUBCLASSE": "Subclasse (tarifa)",
          "DETALHE": "Detalhe (tarifa)", "ACESSANTE": "Acessante (tarifa)",
          "CHAVE_E": "Chave energia", "REGRA_E": "Regra da chave de energia",
          "CHAVE_D": "Chave demanda", "REGRA_D": "Regra da chave de demanda",
          "E_TUSD_MWH": "Energia TUSD (MWh)", "E_TE_MWH": "Energia TE (MWh)",
          "DEMANDA_KW": "Demanda (kW)", "DESC_DEM_PCT": "Desconto demanda % (SAMP)",
          "DESC_E_PCT": "Desconto energia TUSD % (SAMP)",
          "RECEITA_BRL": "RECEITA SAMP", "BLOCO1_TA": "VALOR CALCULADO (bloco 1)",
          "DIFERENCA_BRL": "DIFERENÇA (R$)", "PCT_AJUSTE": "DIFERENÇA (%)",
          "TARIFAVEL_TXT": "Tarifável?", "REGIME": "Regime de ajuste",
          "DESC_FIO": "% de desconto no fio", "AJUSTE_1A_BRL": "Ajuste 1ª passada (R$)",
          "AJUSTE_2A_BRL": "Ajuste 2ª passada (R$)",
          "RESIDUO_NAO_AJUSTAVEL": "Resíduo não ajustável (R$)"}

ordem_opcao = {o: i for i, o in enumerate(["CATIVO", "DISTRIBUIÇÃO", "GERAÇÃO", "LIVRE", "SUPRIMENTO"])}
ordem = base.assign(_o=base["OPCAO"].map(ordem_opcao).fillna(99)).sort_values(
    ["_o", "SUBGRUPO", "MODALIDADE", "TIPO_MERCADO", "SUBCLASSE_SAMP", "DETALHE_SAMP", "POSTO"]).index


def familias_por_camada(bloco):
    '''Sum the components of each family, in each layer.'''
    return pd.DataFrame({f"{cam} · {fam}": bloco[cam][lista].sum(axis=1)
                         for cam in CAMADAS for fam, lista in FAMILIA.items()})


def monta_aba(b1, b2, extra=None):
    partes = [base.loc[ordem, IDENTIFICACAO].reset_index(drop=True),
              b1.loc[ordem].add_prefix("[1] ").reset_index(drop=True),
              base.loc[ordem, ENTRE_BLOCOS].reset_index(drop=True),
              b2.loc[ordem].add_prefix("[2] ").reset_index(drop=True)]
    if extra is not None:
        partes.append(extra.loc[ordem].add_prefix("[2] ").reset_index(drop=True))
    return pd.concat(partes, axis=1).rename(columns=ROTULO)


aba_componentes = monta_aba(BLOCO1["TA"][COMPONENTES], BLOCO2["TA"][COMPONENTES],
                            extra=pd.DataFrame({f"Total {c}": BLOCO2[c][COMPONENTES].sum(axis=1)
                                                for c in CAMADAS}))
aba_familias = monta_aba(familias_por_camada(BLOCO1), familias_por_camada(BLOCO2))


def formata(planilha, n_ident):
    planilha.freeze_panes = planilha.cell(row=2, column=n_ident + 1)
    for celula in planilha[1]:
        celula.font = Font(bold=True, size=9, color="FFFFFF")
        celula.fill = PatternFill("solid", fgColor="1F4E79")
        celula.alignment = Alignment(horizontal="center", wrap_text=True, vertical="center")
    planilha.row_dimensions[1].height = 42
    pct = {ROTULO[c] for c in ("PCT_AJUSTE", "DESC_FIO", "DESC_DEM_PCT", "DESC_E_PCT")}
    for col in range(1, planilha.max_column + 1):
        letra = get_column_letter(col)
        titulo = planilha.cell(row=1, column=col).value
        planilha.column_dimensions[letra].width = 22 if col <= n_ident else 16
        numerica = col > IDENTIFICACAO.index("REGRA_D") + 1
        if numerica:
            # SAMP discount rates come in percent points; the others are fractions
            fmt = "0.00%" if titulo in pct and titulo not in (ROTULO["DESC_DEM_PCT"], ROTULO["DESC_E_PCT"]) \
                else "#,##0.00"
            for linha in range(2, planilha.max_row + 1):
                planilha.cell(row=linha, column=col).number_format = fmt
    planilha.auto_filter.ref = f"A1:{get_column_letter(planilha.max_column)}{planilha.max_row}"


SAIDA.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(SAIDA, engine="openpyxl") as escritor:
    aba_componentes.to_excel(escritor, sheet_name="Componentes", index=False)
    aba_familias.to_excel(escritor, sheet_name="Familias", index=False)
    for nome in ("Componentes", "Familias"):
        formata(escritor.book[nome], len(IDENTIFICACAO))

print(f"aba Componentes: {aba_componentes.shape[0]} linhas x {aba_componentes.shape[1]} colunas")
print(f"aba Familias   : {aba_familias.shape[0]} linhas x {aba_familias.shape[1]} colunas")
print(f"gravado: {SAIDA}")
print(f"referencia {REFERENCIA} | " + ", ".join(f"{a} ({100 * p:.1f}%)" for a, p in VIGENCIAS_NO_MES))
if len(RECEITA_REPETIDA):
    print(f"ATENCAO: receita repetida do mes anterior em {len(RECEITA_REPETIDA)} grupo(s) - ver Passo 2b")